In [1]:
chr(0)

'\x00'

In [5]:
print(chr(0))

 


In [7]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
 return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [8]:
class Demo:
    value = 10

    @classmethod
    def add_cls(cls, x):
        return cls.value + x        # ✅ 可以访问 cls.value

    @staticmethod
    def add_static(x):
        return x + 5                # ❌ 不能访问 Demo.value

In [10]:
print(Demo.add_cls(5))

15


In [36]:
def sub_func():
    return [1, 2, 3]  # 普通返回

def main_gen():
    yield sub_func()  # ✅ 可以！因为 list 是可迭代的

for i in main_gen():
    print(i)

[1, 2, 3]


In [1]:
from torch import nn
import torch
class Demo(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.linear=nn.Linear(d_in,d_out)
    def forward(self,in_features):
        return self.linear(in_features)

num=torch.randn(5,10)
A=Demo(10,5)
B=A(num)
print(type(B))

<class 'torch.Tensor'>


In [43]:
import torch
A=torch.range(1,27)
A=A.unsqueeze(0)
A=A.view(3,3,3)
B=torch.tensor([2]*3)
print(A)
A=A*B
print(A)

tensor([[[ 1.,  2.,  3.],
         [ 4.,  5.,  6.],
         [ 7.,  8.,  9.]],

        [[10., 11., 12.],
         [13., 14., 15.],
         [16., 17., 18.]],

        [[19., 20., 21.],
         [22., 23., 24.],
         [25., 26., 27.]]])
tensor([[[ 2.,  4.,  6.],
         [ 8., 10., 12.],
         [14., 16., 18.]],

        [[20., 22., 24.],
         [26., 28., 30.],
         [32., 34., 36.]],

        [[38., 40., 42.],
         [44., 46., 48.],
         [50., 52., 54.]]])


C:\Users\谌伦杰\AppData\Local\Temp\ipykernel_70412\2561022888.py:2: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  A=torch.range(1,27)


In [102]:
import numpy
A=torch.randint(0,10,(5,))
print(A)
B=torch.randint(0,10,(5,))
print(B)
C=torch.stack((A,B),dim=-1)
print(C)
print(A)


tensor([8, 7, 8, 7, 1])
tensor([4, 7, 6, 9, 4])
tensor([[8, 4],
        [7, 7],
        [8, 6],
        [7, 9],
        [1, 4]])
tensor([8, 7, 8, 7, 1])


In [103]:
import torch
import torch.nn as nn

def precompute_freqs_cis(dim: int, max_seq_len: int, theta: float = 10000.0):
    """
    预计算旋转频率的 cos 和 sin 值
    dim: 注意力头的维度 (head_dim)
    max_seq_len: 最大序列长度
    theta: 基础频率，默认 10000.0 (LLaMA 等模型的默认值)
    """
    # 1. 计算 theta_i: [dim/2]
    # 公式: theta_i = 10000 ^ (-2i/d) => exp(-2i/d * ln(10000))
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))

    # 2. 生成位置索引 m: [max_seq_len]
    t = torch.arange(max_seq_len, dtype=torch.float32)

    # 3. 计算 m * theta_i: 使用外积得到矩阵形状 [max_seq_len, dim/2]
    freqs = torch.outer(t, freqs)

    # 4. 复制两份以匹配原始维度 d: [max_seq_len, dim]
    # 这里对应把前半部分和后半部分结合的写法
    freqs = torch.cat((freqs, freqs), dim=-1)

    # 返回 cos 和 sin (方便后续直接相乘)
    return freqs.cos(), freqs.sin()

def rotate_half(x):
    """
    将向量的后半部分取负，并与前半部分交换位置。
    用于实现公式中的: [-x1, x0] (这里 x 已经被分成了两半)
    """
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, position_ids):
    """
    将 RoPE 应用到 Query 和 Key 上
    q, k 形状: [batch_size, num_heads, seq_len, head_dim]
    """
    # 根据 position_ids 提取对应的 cos 和 sin
    # squeeze(1) 处理是因为 position_ids 通常有 batch 维度，我们通过 unsqueeze 对齐维度
    cos = cos[position_ids].unsqueeze(1)  # [batch_size, 1, seq_len, head_dim]
    sin = sin[position_ids].unsqueeze(1)  # [batch_size, 1, seq_len, head_dim]

    # 核心公式：x * cos(m * theta) + rotate_half(x) * sin(m * theta)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)

    return q_embed, k_embed

# ================= 测试代码 =================
if __name__ == "__main__":
    batch_size = 2
    num_heads = 8
    seq_len = 10
    head_dim = 64
    max_seq_len = 1024

    # 1. 预计算频率 (通常在模型初始化时计算一次即可)
    cos, sin = precompute_freqs_cis(head_dim, max_seq_len)

    # 2. 模拟输入的 Query 和 Key
    q = torch.randn(batch_size, num_heads, seq_len, head_dim)
    k = torch.randn(batch_size, num_heads, seq_len, head_dim)
    print(cos.shape)

    # 3. 模拟位置 ID (通常是 0, 1, 2, ..., seq_len-1)
    position_ids = torch.arange(seq_len).unsqueeze(0).expand(batch_size, -1)

    # 4. 应用 RoPE
    q_out, k_out = apply_rotary_pos_emb(q, k, cos, sin, position_ids)

    print("输入 q shape:", q.shape)
    print("输出 q shape:", q_out.shape)
    print("应用 RoPE 成功！")


torch.Size([1024, 64])
输入 q shape: torch.Size([2, 8, 10, 64])
输出 q shape: torch.Size([2, 8, 10, 64])
应用 RoPE 成功！


In [113]:
torch.tril(torch.ones(3, 3, dtype=torch.bool))

tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])

In [119]:

# print(type(B))

RuntimeError: Only Tensors of floating point and complex dtype can require gradients